# שבוע 8: מתארים פורייה אליפטיים (EFA) — מבוא

בשיעור זה נלמד:
- מה זה EFA ולמה הוא שונה מציוני דרך
- כיצד קו מתאר הופך לסדרת הרמוניות
- כיצד לחלץ מתארי EFA עם pyefd
- שחזור צורה ממספרים שונים של הרמוניות

> **הוראות**: הריצו כל תא בסדר מלמעלה למטה.

In [ ]:
!pip install pyefd python-bidi -q
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

## EFA — הרעיון הבסיסי

EFA (Elliptic Fourier Analysis) מפרק כל קו מתאר לסכום של אליפסות בתדרים שונים:

- **הרמוניה 1**: הצורה הכוללת (גדולה, אליפסה עיקרית)
- **הרמוניות 2–3**: פרטים גדולים (זוויות, בליטות)
- **הרמוניות 4+**: פרטים קטנים ועדינים

כל הרמוניה מוגדרת ע"י **4 מקדמים** (a, b, c, d) — לסה"כ **4n מקדמים** עבור n הרמוניות.

In [ ]:
import pyefd

# יצירת צורה סינתטית — אליפסה עם בליטה
theta = np.linspace(0, 2*np.pi, 300, endpoint=False)
x = 2.0 * np.cos(theta) + 0.4 * np.cos(3*theta)
y = 1.0 * np.sin(theta) + 0.25 * np.sin(2*theta)
contour = np.column_stack([x, y])

# חילוץ מקדמי EFA (20 הרמוניות)
coeffs = pyefd.elliptic_fourier_descriptors(contour, order=20, normalize=True)
print(f'מקדמי EFA: {coeffs.shape} (20 הרמוניות × 4 מקדמים)')
print(f'הרמוניה 1: a={coeffs[0,0]:.3f}, b={coeffs[0,1]:.3f}, c={coeffs[0,2]:.3f}, d={coeffs[0,3]:.3f}')

# ציור הצורה המקורית
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(contour[:, 0], contour[:, 1], 'steelblue', linewidth=2)
ax.set_aspect('equal')
ax.set_title(rtl('צורה סינתטית לדוגמה'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## שחזור ממספרים שונים של הרמוניות

כמה הרמוניות מספיקות? בואו נראה:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
orders = [1, 2, 4, 8, 20]

for ax, order in zip(axes, orders):
    reconstructed = pyefd.reconstruct_contour(coeffs, locus=(0, 0),
                                               num_points=300, harmonic=order)
    ax.plot(reconstructed[:, 0], reconstructed[:, 1], 'steelblue', linewidth=2)
    ax.plot(contour[:, 0], contour[:, 1], 'gray', linewidth=1, alpha=0.4, linestyle='--')
    ax.set_aspect('equal')
    ax.set_title(rtl(f'{order} הרמוניות'), fontsize=11)
    ax.set_xlim(-2.8, 2.8)
    ax.set_ylim(-1.6, 1.6)
    ax.axis('off')

plt.suptitle(rtl('שחזור קו מתאר ממספרים שונים של הרמוניות'), fontsize=13)
plt.tight_layout()
plt.show()

## גרזני כנפיים — הנתונים שלנו

גרזני הכנפיים של תקופת הברונזה הבינונית כוללים שתי קבוצות עיצוביות:
- **G3**: גרזנים קדומים יותר
- **G4**: גרזנים מאוחרים יותר

נטען נתוני EFA שחולצו מראש.

In [ ]:
import urllib.request

def load_efa_csv(url):
    """טעינת קובץ CSV עם מקדמי EFA ותוויות קבוצה"""
    with urllib.request.urlopen(url) as r:
        lines = r.read().decode('utf-8').strip().split('\n')
    header = lines[0].split(',')
    data, groups = [], []
    for line in lines[1:]:
        parts = line.strip().split(',')
        groups.append(parts[0])
        data.append([float(x) for x in parts[1:]])
    return np.array(data), np.array(groups)

base = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/axes/'
try:
    efa_data, groups = load_efa_csv(base + 'axes_efa.csv')
    print(f'נטענו {len(efa_data)} גרזנים, {efa_data.shape[1]} מקדמי EFA')
    print(f'G3: {np.sum(groups=="G3")}, G4: {np.sum(groups=="G4")}')
except Exception as e:
    print(f'משתמשים בנתוני דוגמה: {e}')
    np.random.seed(42)
    n_g3, n_g4 = 30, 35
    base_g3 = np.random.randn(40) * 0.1
    base_g4 = base_g3 + np.random.randn(40) * 0.05 + 0.15
    efa_data = np.vstack([
        base_g3 + np.random.randn(n_g3, 40) * 0.08,
        base_g4 + np.random.randn(n_g4, 40) * 0.08
    ])
    groups = np.array(['G3'] * n_g3 + ['G4'] * n_g4)
    print(f'נוצרו נתוני דוגמה: {efa_data.shape}')

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
scores = pca.fit_transform(efa_data)
var = pca.explained_variance_ratio_ * 100

print('שונות מוסברת:')
for i in range(5):
    print(f'  PC{i+1}: {var[i]:.1f}%')

colors = {'G3': '#9C27B0', 'G4': '#4CAF50'}
fig, ax = plt.subplots(figsize=(9, 7))

for group, color in colors.items():
    mask = groups == group
    ax.scatter(scores[mask, 0], scores[mask, 1], c=color, s=80, alpha=0.8,
               label=f'{group} (n={mask.sum()})', edgecolors='white')

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(rtl(f'PC1 ({var[0]:.1f}% שונות)'))
ax.set_ylabel(rtl(f'PC2 ({var[1]:.1f}% שונות)'))
ax.set_title(rtl('מרחב EFA — גרזני כנפיים G3 ו-G4'), fontsize=13)
ax.legend(title=rtl('קבוצה עיצובית'))
plt.tight_layout()
plt.show()

## תרגיל

1. שנו את `order` ב-5 לספרות שונות — כמה הרמוניות מספיקות לשחזור סביר?
2. מה ההבדל בין EFA לציוני דרך? מה יתרון כל שיטה?
3. האם G3 ו-G4 נפרדים בבירור במרחב PCA? לאיזה ציר בעיקר?